# Labeling with LLM

<a target="_blank" href="https://colab.research.google.com/github/zentralbibliothek-zuerich/zblab-summerschool-2026/blob/main/notebooks/06_labeling_with_llm.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This notebook contains boilerplate code that you don't need to interact with directly.

The sections where you can safely experiment or customize are clearly marked with such comments:

```python
# ⬇️⬇️⬇️
YOUR_INPUT = ""
# ⬆️⬆️⬆️

## Setup

### Housekeeping (no interaction required)

In [ ]:
%pip install openrouter

In [ ]:
import json
import os
import random
import textwrap
import time
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.notebook import tqdm
from openrouter import OpenRouter

tqdm.pandas()

In [ ]:
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
DATA_DIR = Path('/content/drive/MyDrive/ZBLabSummerSchool2026/data') if IN_COLAB else Path('../data')
MODEL_NAME = "openai/gpt-5.4-mini"

In [ ]:
def confirm(question: str = "Do you want to execute this cell?") -> bool:
    """Ask for a yes/no confirmation before running a step.

    Args:
        question: Prompt shown to the user.

    Returns:
        True for yes, False for no.
    """
    while True:
        response = input(f"{question} (y/n): ").lower()
        if response in ["y", "yes"]:
            return True
        elif response in ["n", "no"]:
            return False
        else:
            print("Please enter 'y' or 'n'.")

### Setup (interaction required)

In [ ]:
### ⬇️⬇️⬇️  Adjust here if you want to continue with your own query
CORPUS_NAME = "armenpflege"
USE_YOUR_DATA = False # Set to True if you want to use your own data
YOUR_NAME = "niclas"
### ⬆️⬆️⬆️

### Load the data

In [ ]:
if IN_COLAB and USE_YOUR_DATA: #confirm("Do you want to mount your Google Drive?"):
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DATA_DIR, exist_ok=True)

#### <img src="https://cdn.svglogos.dev/logos/google-drive.svg" alt="💾" width=16> Load your own data from Google Drive

In [ ]:
# ⬇️⬇️⬇️
# Separator for the labeled csv-file.
# Excel likes to use ";" as a separator, even when explicitly exporting to CSV.
# You can inspect your labeled data file to check which separator is used, and adjust this variable accordingly.
SEPARATOR = ";" 
# ⬆️⬆️⬆️

if USE_YOUR_DATA:
    # RAWDATA_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.parquet" # Use data from filtering module
    # raw_df = pd.read_parquet(RAWDATA_PATH)

    # SENTENCES_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.sentences.parquet" # Use data from filtering module
    # sentences_df = pd.read_parquet(SENTENCES_PATH)

    # raw_df = raw_df.join(sentences_df, on="id")

    LABELS_PATH = DATA_DIR / f"{CORPUS_NAME}.filtered.pp.label.{YOUR_NAME}.csv"
    labels_df = pd.read_csv(LABELS_PATH, index_col="id", sep=SEPARATOR)

### <img src="https://www.zb.uzh.ch/themes/zb/assets/images/favicon-192.png" alt="💾" width=16> Load the 'Armenpflege' example

In [ ]:
if not USE_YOUR_DATA:
    LABELS_PATH = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.niclas.csv"
    labels_df = pd.read_csv(LABELS_PATH, index_col="id", sep=";")

### Enable LLM interaction

We use Google Colab's secret manager to store the OpenRouter API key securely. To add your key, click the key icon 🗝️ in the left sidebar, then select `Add new secret`. Enter a name for your secret, e.g. `openrouter-api-key`, and paste your API key into the value field. Use this secret name in the code cell below. 

![colab-secrets](../assets/colab_secrets_1.png)

After adding your key, allow notebook access:

![notebook-access](../assets/colab_secrets_2.png)

We are now ready to make our first request to the LLM!

In [ ]:
from google.colab import userdata
# ⬇️⬇️⬇️ 
API_KEY = userdata.get("openrouter-api-key")  # Replace with the name of your secret in Google Colab
# ⬆️⬆️⬆️

In [ ]:
from openrouter import OpenRouter

with OpenRouter(
  api_key=API_KEY,
) as client:
  response = client.chat.send(
    model=MODEL_NAME,
    messages=[
      {
        "role": "user",
        "content": "What is the meaning of life? Be short and concise in your answer."
      }
    ],
    max_tokens=100
  )

print(response.choices[0].message.content)

## Introduction to Prompting

### Hands-on:
Now try the following things:

1.   Change the user prompt and re-run the request.
2.   Change the system prompt to something completely random and re-run the request.

#### Temperature and top-p
Let's go through each of these arguments one-by-one. Below you will see API calling code for the LLM with a slightly more complex prompt. Try out requests to this LLM where you:


1.   Vary the temperature parameter.
2. Vary the top-p parameter.

Your task is try to get the funniest and weirdest response possible from the model.

In order to prevent too excessive an output (and to protect our budget!) we'll limit the output to a maximum of 50 tokens using the `max_tokens` argument; please do not change this.

In [ ]:
# ⬇️⬇️⬇️ Adjust the system and user prompts here to see how it affects the output
longer_system_prompt = (
    "You are a political text classifier. Given a social media post, return a JSON object "
    "with exactly four fields: "
    "\"topic\" (a short label for the main topic, e.g. 'economy', 'healthcare', 'immigration'), "
    "\"sentiment\" (one of: 'positive', 'negative', 'neutral'), and "
    "\"is_political\" (true or false). "
    "Return only the JSON object, nothing else."
)
longer_user_prompt = (
    "Post: 'Can't believe they're cutting the healthcare budget again while giving tax breaks "
    "to billionaires. This government doesn't care about ordinary people.'"
)
# ⬆️⬆️⬆️

In [ ]:
with OpenRouter(
  api_key=API_KEY,
) as client:
    response = client.chat.send(
        model=MODEL_NAME,
        messages=[
        {"role": "system", "content": longer_system_prompt},
        {"role": "user", "content": longer_user_prompt}
        ],
        max_tokens=50,

        # ⬇️⬇️⬇️ Adjust these parameters to see how it affects the output
        temperature=1, # 0-1
        top_p=0.1 # 0-1
        # ⬆️⬆️⬆️
    )

output_text = response.choices[0].message.content  # take the output and extract the text response only

# parse and pretty-print the JSON output
parsed = json.loads(output_text)
print(json.dumps(parsed, indent=2))

## Check your labels

### Inspect your gold standard

For our 'Armenpflege' example, we inductively arrived at three classes for the articles.

![categories_better](../assets/armenpflege_kategoriensystem.excalidraw.png)

| Label                    | Description                                                                                                                                                                                                                                     |
| ------------------------ | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| *Meinungsartikel*        | An article where someone (the author or a cited speaker) explicitely mentions how the 'Armenpflege' or a related topic ought to be or where an explicit valorisation is made.                                                                   |
| *Unspezifischer Bericht* | An article containing a recollection of events or a description of facts without any explicit markers of subjectivity. Does not include articles that fall under "Einzelfalldarstellung".                                                       |
| *Einzelfalldarstellung*  | An article that describes the circumstances of a concrete person or group that interacted with the 'Armenpflege', mainly as recipients. Articles around singular politicians or institutional actors do not fall under 'Einzelfalldarstellung'. |

These categories are partially driven by the hypothesis that the share of 'Meinungsartikel' becomes smaller and smaller as time progresses and the institution of "Armenpflege" changes from novelty to normalcy.


In [ ]:
labels_df[labels_df["label"].notna()].value_counts(subset=["label"])

### Assess labels and classification scheme by comparing the results of two human coders

You want to use these metrics and evaluations during the annotation cycle, to refine your understanding of the categories and your code book.

This ultimately serves in creating an *intersubjective* and *reliable* coding of your data.

In [ ]:
NICLAS_PATH = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.niclas.csv"
niclas_df = pd.read_csv(NICLAS_PATH, index_col="id", sep=";")
niclas_df = niclas_df[niclas_df["label"].notna()]

ELIAS_PATH = "https://media.githubusercontent.com/media/zentralbibliothek-zuerich/zblab-summerschool-2026/main/data/armenpflege.filtered.pp.label.elias.csv"
elias_df = pd.read_csv(ELIAS_PATH, index_col="id", sep=";")
elias_df = elias_df[elias_df["label"].notna()]

# Only keep the entries that both annotators labeled
common_ids = sorted(set(niclas_df.index) & set(elias_df.index))
niclas_df = niclas_df.loc[common_ids]
elias_df = elias_df.loc[common_ids]

print(f"Number of the same datapoints labeled by both annotators: {len(common_ids)}")

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score, matthews_corrcoef
from matplotlib import pyplot as plt
import seaborn as sns

def evaluate_rater_agreement(
    labels_1: pd.Series,
    name_1: str,
    labels_2: pd.Series,
    name_2: str,
) -> None:
    """Plot and print agreement metrics for two sets of labels.

    Args:
        labels_1: First label series.
        name_1: Display name for first rater.
        labels_2: Second label series.
        name_2: Display name for second rater.
    """
    mask = labels_1.notna() & labels_2.notna()
    labels_1 = labels_1[mask]
    labels_2 = labels_2[mask]

    labels = list(set(labels_1) | set(labels_2))

    cm = confusion_matrix(labels_1, labels_2, labels=labels)
    kappa = cohen_kappa_score(labels_1, labels_2, labels=labels)
    mcc = matthews_corrcoef(labels_1, labels_2)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
    plt.xlabel(name_2)
    plt.ylabel(name_1)
    plt.title('Confusion Matrix')
    plt.show()
    print(f"Cohen's Kappa: {kappa:.4f}")
    print(f"Matthews Correlation Coefficient: {mcc:.4f}")

evaluate_rater_agreement(niclas_df["label"], "Human Rater Niclas", elias_df["label"], "Human Rater Elias")

In [ ]:
# Show disagreemnts
disagreement_df = pd.merge(niclas_df, elias_df["label"], left_index=True, right_index=True, suffixes=("_niclas", "_elias"))
disagreement_df = disagreement_df[disagreement_df["label_niclas"] != disagreement_df["label_elias"]]


In [ ]:
row = disagreement_df.sample(n=1).iloc[0]
print(f"ID:                  {row.name}")
print(f"Label from Elias:    {row['label_elias']}")
print(f"Label from Niclas:   {row['label_niclas']}", end="\n\n")
print(f"Article Pseudo-Paragraph:")

word_to_highlight = "Armenpflege"
text = row['pseudo_paragraph'].replace(word_to_highlight, f"\033[1m{word_to_highlight}\033[0m")

print(textwrap.fill(text, width=80))

## Develop an LLM Labeler

### Split the labeled data

We split the labeled data into two smaller datasets:
- tuning set
- evaluation set

We use the tuning set to repeatedly query the LLM and 'tune' our prompt.
If something gets misclassified with the tuning set, we try to steer the LLM in a better direction.
However, we need to remain general enough in our prompt so that we are not only optimizing for the tuning set, but for all data.

Once we are happy with the results on the tuning set, we evaluate **once** with the evaluation set and don't change the prompt any further!

The resulting metrics give us an impression for how well the model will behave on the whole dataset.

In [ ]:
# Create two columns for the LLM responses
labels_df["predicted_label"] = None
labels_df["reasoning"] = None

# Split the dataframe into labeled and unlabeled subsets
unlabeled_df = labels_df[labels_df["label"].isna()].copy() 
gold_df = labels_df[~labels_df["label"].isna()].copy()

In [ ]:
from sklearn.model_selection import train_test_split

sample_size = len(gold_df)
evaluation_size = min(20, sample_size // 2)
tuning_size = sample_size - evaluation_size

sample_df = gold_df.sample(sample_size, random_state=42)
evaluation_df, tuning_df = train_test_split(
    sample_df,
    test_size=tuning_size,
    random_state=42,
    # By using stratify, we ensure that the label distribution in the evaluation 
    # and tuning sets is similar to the original distribution.
    stratify=sample_df["label"],
)

print("Evaluation set label distribution:")
print(evaluation_df["label"].value_counts())
print()

print("Tuning set label distribution:")
print(tuning_df["label"].value_counts())

### Define the labels for the LLM

![categories_better](../assets/armenpflege_kategoriensystem.excalidraw.png)

In [ ]:
# ⬇️⬇️⬇️ Adjust the categories according to your data
categories = {
    "Meinungsartikel": "The article expresses a subjective opinion or stance or desired actions regarding the 'Armenpflege'. This can be the opinion of the author or of other individuals mentioned in the article. Reports on political discussions are included, if the stances are portrayed. Meinungsartikel may be emotionally charged but can also be very neutral and argumentative.",
    "Einzelfalldarstellung": "The article describes a specific individual case of a person or a group of people that interacted with the 'Armenpflege', predominantly recipients of charity. The mention of singular politicians or speakers or political acts is no Einzelfalldarstellung.",
    "Unspezifischer Bericht": "The article is an objective report that does not fall under the other categories. It is most often about the outcomes of political events, and it is the most common category in the data.",
}
# ⬆️⬆️⬆️

## Few-Shot Prompting

Copy the output of the cell below into the next cell and fill in examples from your data.

In [ ]:
obj = {category: "   " for category in categories.keys()}
print("FEW_SHOT_EXAMPLES = " + json.dumps(obj, indent=2))

In [ ]:
# ⬇️⬇️⬇️ # Copy-paste and fill in here
FEW_SHOT_EXAMPLES = {
  "Meinungsartikel": "Traf und anschaulich der Abschnitt: « Sozialpolitik der Gemeinde », wo es heißt: « Es ist dringend nötig, daß die Beamten mit sozialem Oele gesalbt werden. Sonst knarrt das Räderwerk der Bürokratie in einer unerträglichen Weise. Der Betreibungsbeamte wird die harten Gläubiger gegen den würdigen armen Schuldner zur Milde stimmen... Der Zivilstandsbeamte wird, wenn es nötig und tunlich, die Brautleute zur kirchlichen Trauung bewegen Ein Apostolat von unschätzbarem Werte üben die staatlichen und kommunalen Armenpfleger, zumal durch die Sorge für gute christliche Erziehung der ihnen anvertrauten Kinder. Ebenso die Friedensrichter durch Verhinderung leichtfertiger Prozesse und Schlichtung von Feindschaften. » Hier schreibt Prof. Dr. Beck den Satz, der seine eigene Schriftstellertätigkeit erklärt: « Sei also der Sämann, erfasse das herrliche Saatgut lichtvoller und lebensspendender katholischer Volksschriften und wirf sie mit kräftiger Hand hinein in die Furchen der Zeit und des heutigen Volkslebens!» So wurde denn Prof. Beck, ungeachtet seines Lehrstuhles an der Universität, « der schweizerische Alban Stolz im Volkskalender für Freiburg und Wallis. » Schon im ersten Jahrgang 1910 eröffnete sich Professor Beck hier eine richtige Bauernkanzel.",
  "Einzelfalldarstellung": ". Charles Bilat, durch Strafverfügung der eidgenössischen Oberzolldirektion vom 17, Mai 1948 zu einer Busse von Fr. 406.67 verurteilt, unter Nachlass eines Bussendrittels wegen vorbehaltloser Unterziehung, weil er 2000 Heftchen Zigarettenpapier gekauft und weiterveräussert hatte, obwohl er wusste, dass dasselbe aus Frankreich in die Schweiz eingeschmuggelt worden war, Der Verurteilte ersucht um Brlass der Busse, die er nicht zahlen könne. Er sei Vater von vier minderjährigen Kindern, und sein Einkommen als einfacher Handlanger sei bescheiden. Sollte die Busse in Haft umgewandelt werden, so müsste seine Familie durch die Armenpflege unterstützt werden. Die eidgenössische Oberzolldirektion beantragt, das Gesuch zurzeit abzuweisen. Demgegenüber beantragen wir die Herabsetzung der Busse bis zu Fr. 100, weil Bilat gut beleumdet ist und nachgewiesenermassen in ärmlichen Verhältnissen lebt. Ein weiteres Entgegenkommen können wir indessen nicht befürworten.",
  "Unspezifischer Bericht": "Es werden gewählt: Usteri'Pestlllozzi, Kollbrunner, Dr. Kunz, Zellweger. Dr. Bader, Pflüger, Suter- Thahien, Streuli-tzoen. Präsident: Usteri- P e st a l 0 z z i. Die Armenpflege und die Waisenhauspflege sollen erst neu bestellt werden » ach dem Volksentscheid über die neue Gemeindeordnung. Es werden auch die andern auf der Traktandenliste vorgesehenen Wähle » verschöbe ». Folgenden Bürgerrechtsgesuchen wird ent » sprüche «: Wilhelm Friedr. Bubeck." # NZZ-1907-08-26-a-i0001
}
# ⬆️⬆️⬆️

In [ ]:
# This response format forces the model to always return an expected structure
response_format = {
            "type": "json_schema",
            "json_schema": {
                "name": "armenpflege_classification",
                "strict": True,
                "schema": {
            "type": "object",
            "properties": {
                # Reasoning needs to be generated before the label.
                # That way, the model can use its own generated reasoning to determine the label.
                "reasoning": {
                    "type": "string"
                },
                "label": {
                    "type": "string",
                    "enum": list(categories.keys())
                },
            },
            "required": ["label", "reasoning"],
            "additionalProperties": False
        }
    }
}

def get_model_response(post: str) -> dict[str, Any]:
    """Run the classifier prompt and return parsed model output.

    Args:
        post: Article text (usually a pseudo-paragraph).

    Returns:
        Parsed JSON object with reasoning and label fields.
    """
    system_prompt = (

        # ⬇️⬇️⬇️ Adjust the system prompt to guide the model's response
        "You are a classifier for historical articles on 'Armenpflege'. "
        "You are given a text excerpt from a historical news page. "
        "The OCR can be faulty, and the text may contain sentences from other articles at the start and end. "
        "Only classify the thematic cluster around 'Armenpflege' in the middle of the text, not the sentences at the start and end. "
        "Your task is to identify the relevant section of the text and classify the article into one of the following categories based on its content and style. "
        # ⬆️⬆️⬆️
        "\n"

        # Define the output format and instructions for the model
        "You must return a JSON object with the following structure with exactly two fields: "
        "\"reasoning\" (a very VERY brief explanation of why the post was classified with the chosen label as opposed to the other labels), "
        "\"label\" (one of: "
        + ", ".join([f"'{cat}' ({desc})" for cat, desc in categories.items()])
        + "), "
        "Return only the JSON object, nothing else."
        "\n"

        # Add examples for few-shot learning
        "Here are some examples of how to classify articles on 'Armenpflege':\n"
        + "\n\n".join([f"Post: '{example}'\nResponse: {json.dumps({'reasoning': '...', 'label': cat}, indent=2)}" for cat, example in FEW_SHOT_EXAMPLES.items()])
    )
    user_prompt = f"Post: '{post}'"

    with OpenRouter(
      api_key=API_KEY,
    ) as client:
        response = client.chat.send(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format=response_format,
            max_tokens=120,
        )

    prediction = json.loads(response.choices[0].message.content)

    return prediction

get_model_response("25 beträgt, wird Kenntnis genommen. An die Kosten der Rodung und Urbarisierung der « Esclhaldc », Gemeinde Muttenz, wird ein Staatsbeitrag bewilligt. Vom vorliegenden Bericht der durchgeführten Gewerbehilfe des Schweizerischen Verbandes der gewerblichen Bürgschaftsgenossenschaften wird Kenntnis genommen. Die Wahlen eines Mitgliedes des Gemeinderates Lausen, eines Mitgliedes der Armenpflege Giebenach und einer Primarlehrcrin für die Gemeinden Arisdorf und Hersberg werden bestätigt. Sanierung des Kirchen- und Schulgutes —st. Seit mehreren Jahren geht das Vermögen des Kirchen- und Schulgutes und damit auch der Zinsertrag unaufhaltsam zurück. Innert zwanzig Jahren hat des Vermögen um 300 911 Fr. abgenommen und betrug 1940 noch 2 986 910 Fr.; an Zinsen stehen 66 572 Fr. weniger zur Verfügung.")

In [ ]:
def safe_get_model_response(article: str) -> pd.Series:
    """Call the model safely and normalize error handling.

    Args:
        article: Article text to classify.

    Returns:
        Series with two fields: predicted label and reasoning/error message.
    """
    try:
        prediction = get_model_response(article)
        label = prediction.get("label", "error")
        reasoning = prediction.get("reasoning", "No reasoning provided.")
        return pd.Series([label, reasoning])
    except Exception as e:
        print(f"Error processing article: {e}")
        return pd.Series(["error", str(e)])

tuning_df[["predicted_label", "reasoning"]] = tuning_df["pseudo_paragraph"].progress_apply(safe_get_model_response)

In [ ]:
tuning_df

In [ ]:
evaluate_rater_agreement(tuning_df["label"], "Human Rater", tuning_df["predicted_label"], "LLM")


#### Inspect disagreements

In [ ]:
tuning_df[tuning_df["label"] != tuning_df["predicted_label"]][["pseudo_paragraph", "label", "predicted_label", "reasoning"]]

In [ ]:
def show_single_inference(df: pd.DataFrame, id: str) -> None:
    """Print one article with human and LLM labels for inspection.

    Args:
        df: Dataframe that contains labels, prediction, and reasoning columns.
        id: Article id to inspect.
    """
    row = df.loc[id]

    print(f"ID:                  {id}")
    print(f"Human Label:         {row['label']}")
    print(f"LLM Predicted Label: {row['predicted_label']}", end="\n\n")
    print(f"Reasoning:")
    print(textwrap.fill(row['reasoning'], width=80), end="\n\n")
    print(f"Article Pseudo-Paragraph:")
    print(textwrap.fill(row['pseudo_paragraph'], width=80))

# ⬇️⬇️⬇️ Adjust the ID to show the inference for a specific article
show_single_inference(tuning_df, "DTT-1962-03-02-a-i0143")
# ⬆️⬆️⬆️

#### 🔄️ Repeat

Now you have determined how the model behaves on the tuning set given your prompt.
You can now rework your prompt, and optimize on the tuning set until you are happy with the results.

⚠️ **But take caution!** If your prompt is too heavily tailored to the tuning set, it will no longer generalize to the whole data!

To test this generalization, we finally evaluate the prompt on the evaluation set.
The results give us an impression on how well the model will behave on unseen data.
*But only if we don't change it anymore after seeing the results on the evaluation set!*

In [ ]:
evaluation_df[["predicted_label", "reasoning"]] = evaluation_df["pseudo_paragraph"].progress_apply(safe_get_model_response)

In [ ]:
evaluate_rater_agreement(evaluation_df["label"], "Human Rater", evaluation_df["predicted_label"], "LLM")

In [ ]:
print("Accuracy:", (evaluation_df["label"] == evaluation_df["predicted_label"]).mean())

## Run inference on full unlabeled set

### Add already inferred labels to the output

We already inferred labels for the articles in the `tuning_df` and `evaluation_df`.
Therefore, we add these labels to the final output.

In [ ]:
if USE_YOUR_DATA:
    INFERENCE_OUTPATH = DATA_DIR / f"{CORPUS_NAME}.filtered.pp.label.llm.csv"
    if INFERENCE_OUTPATH.exists():
        inference_df = pd.read_csv(INFERENCE_OUTPATH, index_col="id", sep=",")
    else:
        inference_df = pd.DataFrame(columns=["predicted_label", "reasoning"])
    
    predicted_tuning_df = tuning_df[tuning_df["predicted_label"].notna()]
    predicted_tuning_df = predicted_tuning_df[["predicted_label", "reasoning"]]

    predicted_evaluation_df = evaluation_df[evaluation_df["predicted_label"].notna()]
    predicted_evaluation_df = predicted_evaluation_df[["predicted_label", "reasoning"]]

    inference_df = pd.concat([inference_df, predicted_tuning_df, predicted_evaluation_df], axis=0)
    inference_df = inference_df[~inference_df.index.duplicated(keep="first")]
    inference_df.to_csv(INFERENCE_OUTPATH, index=True)

### Determine the articles still missing a predicted label

In [ ]:
if USE_YOUR_DATA:
    TIMEOUT = 0  # The response will take long enough, so we can set timeout to 0 to avoid unnecessary waiting time between requests

    def load_missing_inference_ids(print_info: bool = True) -> tuple[list[str], pd.DataFrame]:
        """Load the list of unlabeled items and check which ones still need inference."""
        # if INFERENCE_OUTPATH.exists():
        #     inference_df = pd.read_csv(INFERENCE_OUTPATH, index_col="id")
        # else:
        #     inference_df = pd.DataFrame(columns=["predicted_label", "reasoning"])
        inference_df = pd.read_csv(INFERENCE_OUTPATH, index_col="id")
        already_inferred_ids = set(inference_df.index.to_list())

        # Get IDs that need inference
        missing_ids = [id for id in labels_df.index if id not in already_inferred_ids]
        
        if print_info:
            print(f"Missing {len(missing_ids)} items. (Already inferred {len(already_inferred_ids)})")
            print(f"Requests for missing items will take approximately {len(missing_ids) * max(TIMEOUT, 1) / 60:.2f} minutes (assuming ~{1 / max(TIMEOUT, 1):.2f} request per second).")
        
        return missing_ids, inference_df

    missing_ids, inference_df = load_missing_inference_ids()
    
else:
    print("Skipping loading of existing inference results since USE_YOUR_DATA is set to False.")

In [ ]:
confirm("Do you want to run inference on the full unlabeled set? This will make API calls for each unlabeled item and may take a long time. Do you want to proceed?")  

# Reload missing IDs to ensure we have the latest state
missing_ids, inference_df = load_missing_inference_ids()

try:
    for i, doc_id in enumerate(missing_ids):
        time.sleep(random.uniform(TIMEOUT * 0.5, TIMEOUT * 1.5))
        print(f"({i+1}/{len(missing_ids)}) - Processing document {doc_id}", end="\r")
        
        # Get the text from unlabeled_df
        article_text = unlabeled_df.loc[doc_id, "pseudo_paragraph"]
        
        # Run inference
        prediction = safe_get_model_response(article_text)
        
        # Create a row with the prediction
        row_df = pd.DataFrame({
            "id": [doc_id],
            "predicted_label": [prediction[0]],
            "reasoning": [prediction[1]]
        }).set_index("id")
        
        inference_df = pd.concat([inference_df, row_df], ignore_index=False)
        
        if (i + 1) % 100 == 0:  # Save progress every 100 items
            inference_df.to_csv(INFERENCE_OUTPATH, index=True)
            print(f"Progress saved: {i+1}/{len(missing_ids)} items processed       ")

except KeyboardInterrupt:
    print(f"\nProcess interrupted by user. Saving progress...                              ")
    inference_df.to_csv(INFERENCE_OUTPATH, index=True)
    raise KeyboardInterrupt("Inference process interrupted by user. Progress saved.")
except Exception as e:
    print(f"\nAn error occurred. Saving progress...                              ")
    inference_df.to_csv(INFERENCE_OUTPATH, index=True)
    raise e

# Final save
inference_df.to_csv(INFERENCE_OUTPATH, index=True)
print(f"Inference complete! Results saved to {INFERENCE_OUTPATH}")

### Inspect inference results

In [ ]:
INFERENCE_OUTPATH = DATA_DIR / f"{CORPUS_NAME}.filtered.pp.label.llm.csv"
full_inference_df = pd.read_csv(INFERENCE_OUTPATH, index_col="id")

In [ ]:
print(f"Total inferred items: {len(full_inference_df)}")
print("\nLabel distribution:")
print(full_inference_df["predicted_label"].value_counts())
print(f"\nResults saved to: {INFERENCE_OUTPATH}")

## Show label distribution across time

Only keep articles without errors.

In [ ]:
full_inference_df = full_inference_df.join(labels_df[["year"]], how="left")
full_inference_df = full_inference_df[full_inference_df["predicted_label"] != "error"]

In [ ]:
import matplotlib.pyplot as plt

def plot_predicted_labels_over_time(df: pd.DataFrame, n_years: int) -> None:
    """Visualize absolute and relative label distributions over time.

    Args:
        df: Dataframe with year and predicted_label columns.
        n_years: Width of year buckets.
    """
    df = df.copy()
    df = df[df["year"].notna()].copy()
    df["year"] = df["year"].astype(int)

    start = (df["year"].min() // n_years) * n_years
    end = (df["year"].max() // n_years + 1) * n_years

    # Include the final bin edge so the newest interval is not dropped.
    bins = range(start, end + 1, n_years)
    labels = [f"{bucket_start}-{bucket_start + n_years - 1}" for bucket_start in bins[:-1]]
    df["year_bucket"] = pd.cut(
        df["year"],
        bins=bins,
        labels=labels,
        include_lowest=True,
    )

    label_counts_by_bucket = df.groupby(["year_bucket", "predicted_label"], observed=False).size().unstack(fill_value=0)
    label_counts_by_bucket = label_counts_by_bucket.reindex(labels, fill_value=0)
    label_counts_by_bucket_percentage = label_counts_by_bucket.div(
        label_counts_by_bucket.sum(axis=1).replace(0, pd.NA),
        axis=0,
    ).fillna(0)

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(12, 10),
        sharex=True,
        gridspec_kw={"height_ratios": [1, 2]},
    )

    label_counts_by_bucket.plot(kind="area", stacked=True, ax=axes[0])
    axes[0].set_title("Distribution of Predicted Labels Over Time")
    axes[0].set_xlabel("Year Bucket")
    axes[0].set_ylabel("Number of Articles")
    axes[0].legend(title="Predicted Label")

    label_counts_by_bucket_percentage.plot(kind="area", stacked=True, ax=axes[1])
    axes[1].set_title("Relative Distribution of Predicted Labels Over Time")
    axes[1].set_xlabel("Year Bucket")
    axes[1].set_ylabel("Share of Articles")
    axes[1].legend(title="Predicted Label")

    # Add small percentage annotations inside each stacked area segment.
    frac_values = label_counts_by_bucket_percentage
    for x_pos in range(len(frac_values.index)):
        lower = 0.0
        for col in frac_values.columns:
            value_frac = float(frac_values.iloc[x_pos][col])
            if value_frac <= 0:
                continue
            y_pos = lower + value_frac / 2
            if value_frac >= 0.03:
                axes[1].text(
                    x_pos,
                    y_pos,
                    f"{int(round(value_frac * 100))}%",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="black",
                    alpha=0.8,
                )
            lower += value_frac

    for ax in axes:
        ax.tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

In [ ]:
# ⬇️⬇️⬇️
N_YEARS = 10  # Adjust the bucket size and critique its effect on the visualization.
# ⬆️⬆️⬆️

plot_predicted_labels_over_time(full_inference_df, n_years=N_YEARS)